In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dptel22/firms-h3-daily-features/sih2026_h3_daily_features.parquet


In [2]:
!apt-get update -qq && apt-get install -y osmium-tool
!pip install -q geopandas h3 scikit-learn

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



The following additional packages will be installed:
  libboost-program-options1.74.0
The following NEW packages will be installed:
  libboost-program-options1.74.0 osmium-tool
0 upgraded, 2 newly installed, 0 to remove and 192 not upgraded.
Need to get 882 kB of archives.
After this operation, 3,863 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-program-options1.74.0 amd64 1.74.0-14ubuntu3 [311 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 osmium-tool amd64 1.14.0-1 [571 kB]
Fetched 882 kB in 1s (723 kB/s)
Selecting previously unselected package libboost-program-options1.74.0:amd64.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../libboost-program-options1.74.0_1.74.0-14ub

# 1 Load your real feature table

In [3]:
df_daily = pd.read_parquet("/kaggle/input/datasets/dptel22/firms-h3-daily-features/sih2026_h3_daily_features.parquet")
df_daily["h3_08"] = df_daily["h3_08"].astype(str)
print(f"df_daily: {len(df_daily):,} rows, {df_daily['h3_08'].nunique():,} unique H3 cells")
print(df_daily.columns.tolist())

# Locked geography partitions. These names are used only to define the blind split
# and to constrain TRAIN-only calibration; Test A/B are never used for fitting.
TRAIN_STATES = [
    "Maharashtra", "Karnataka", "Madhya Pradesh",
    "Punjab", "Andhra Pradesh", "Telangana",
]
TEST_A_STATES = ["Gujarat", "Tamil Nadu"]
TEST_B_STATES = ["Jharkhand", "Rajasthan"]

# Locked project CRS for all proximity/distance calculations.
PROJECT_CRS = "EPSG:7755"


df_daily: 1,442,545 rows, 798,705 unique H3 cells
['h3_08', 'acq_date', 'frp_max', 'frp_mean', 'n_detections', 'ti4_max', 'is_saturated_max', 'scan_mean', 'track_mean', 'confidence_high_any', 'pct_high_confidence', 'frp_max_night', 'frp_max_day', 'n_detections_night', 'n_detections_day', 'scan_max', 'track_max', 'daynight', 'satellite_nunique', 'is_static_land', 'is_offshore', 'frp_max_lag7', 'active_days_7d', 'frp_max_lag30', 'active_days_30d', 'active_days_90d', 'acq_month', 'doy_sin', 'doy_cos', 'is_first_observation', 'is_labeled']


# H3 centroid table (computed once per unique cell)

In [4]:
import h3
import geopandas as gpd

unique_h3 = df_daily["h3_08"].unique()
h3_centroids = pd.DataFrame({"h3_08": unique_h3})
h3_centroids[["h3_lat", "h3_lon"]] = h3_centroids["h3_08"].apply(
    lambda c: pd.Series(h3.cell_to_latlng(c))
)
h3_gdf = gpd.GeoDataFrame(
    h3_centroids,
    geometry=gpd.points_from_xy(h3_centroids["h3_lon"], h3_centroids["h3_lat"]),
    crs="EPSG:4326",
)
print(f"H3 centroids: {len(h3_gdf):,}")

H3 centroids: 798,705


# State/UT assignment

In [5]:
import subprocess
import geopandas as gpd

STATE_SHP_BASE = "https://raw.githubusercontent.com/AnujTiwari/India-State-and-Country-Shapefile-Updated-Jan-2020/master/India_State_Boundary"
for ext in ["shp", "shx", "dbf", "prj", "cpg"]:
    subprocess.run(
        ["wget", "-q", f"{STATE_SHP_BASE}.{ext}",
         "-O", f"/kaggle/working/state_boundary.{ext}"],
        check=True,
    )

states = gpd.read_file("/kaggle/working/state_boundary.shp").to_crs(PROJECT_CRS)
states["State_Name"] = states["State_Name"].replace(
    {
        "Chhattishgarh": "Chhattisgarh",
        "Telengana": "Telangana",
        "Tamilnadu": "Tamil Nadu",
    }
)
states = states.dissolve(by="State_Name", as_index=False)
states["geometry"] = states["geometry"].buffer(0)

assert len(states) == 36, f"Expected 36 states/UTs, got {len(states)}"
print(f"States loaded: {len(states)}")

# State assignment is geometric only. Use projected CRS for the predicate and
# for the rare nearest-state resolution path.
h3_projected = h3_gdf.to_crs(PROJECT_CRS)

joined = gpd.sjoin(
    h3_projected,
    states[["State_Name", "geometry"]],
    how="left",
    predicate="within",
)
joined = joined.drop(columns="index_right").rename(columns={"State_Name": "state"})
joined["state_assignment_method"] = np.where(
    joined["state"].notna(), "within", "needs_nearest_resolution"
)

dup_mask = joined["h3_08"].duplicated(keep=False)
dup_ids = joined.loc[dup_mask, "h3_08"].unique()
unmatched_ids = joined.loc[joined["state"].isna(), "h3_08"].unique()
needs_resolution = np.unique(np.concatenate([dup_ids, unmatched_ids]))

print(
    f"Cells matching >1 state via 'within': {len(dup_ids):,} "
    f"out of {joined['h3_08'].nunique():,}"
)
print(
    f"Cells with no 'within' state match: {len(unmatched_ids):,}"
)

if len(needs_resolution) > 0:
    resolve_points = joined.loc[
        joined["h3_08"].isin(needs_resolution),
        ["h3_08", "h3_lat", "h3_lon", "geometry"],
    ].drop_duplicates("h3_08").copy()

    resolved = gpd.sjoin_nearest(
        resolve_points,
        states[["State_Name", "geometry"]],
        how="left",
        distance_col="_state_distance_m",
    )
    resolved = resolved.rename(columns={"State_Name": "state"})

    # Exact boundary ties can still return multiple states. Resolve them
    # deterministically by shortest projected distance, then state name.
    resolved = (
        resolved
        .sort_values(["h3_08", "_state_distance_m", "state"], kind="stable")
        .drop_duplicates("h3_08", keep="first")
    )
    resolved["state_assignment_method"] = np.where(
        resolved["_state_distance_m"].eq(0),
        "nearest_boundary_tie_break",
        "nearest_unmatched",
    )
    resolved["_state_distance_km"] = resolved["_state_distance_m"] / 1000.0

    # Rebuild the non-resolved rows without duplicate boundary matches.
    clean = joined[~joined["h3_08"].isin(needs_resolution)].copy()
    clean["_state_distance_km"] = 0.0

    joined = pd.concat(
        [
            clean,
            resolved[
                [
                    "h3_08", "h3_lat", "h3_lon", "geometry",
                    "state", "state_assignment_method", "_state_distance_km"
                ]
            ],
        ],
        ignore_index=True,
    )

h3_gdf = gpd.GeoDataFrame(joined, geometry="geometry", crs=PROJECT_CRS)

assert h3_gdf["h3_08"].is_unique, "State assignment left duplicated H3 cells."
assert h3_gdf["state"].notna().all(), (
    "State assignment left unmatched H3 cells after nearest-state resolution."
)
assert h3_gdf["state"].nunique() <= len(states), "More assigned states than state geometries."
assert set(h3_gdf["state"].dropna().unique()).issubset(set(states["State_Name"])), "Unknown state value produced."

print(
    f"Resolved exactly one state for {h3_gdf['h3_08'].nunique():,} "
    f"unique H3 cells."
)
print(
    "Nearest-resolution distances (km) among non-within cells: "
    f"{h3_gdf.loc[h3_gdf['state_assignment_method'] != 'within', '_state_distance_km'].describe().to_dict()}"
)


States loaded: 36
Cells matching >1 state via 'within': 12 out of 798,705
Cells with no 'within' state match: 1,657
Resolved exactly one state for 798,705 unique H3 cells.
Nearest-resolution distances (km) among non-within cells: {'count': 1669.0, 'mean': 1.149580642312871, 'std': 3.5602217345773313, 'min': 0.0, '25%': 0.2644696461457802, '50%': 0.7349562873968687, '75%': 1.5282519970139383, 'max': 82.35081742843094}


# WRI Global Power Plant Database

In [6]:
from sklearn.neighbors import NearestNeighbors

wri = pd.read_csv(
    "https://raw.githubusercontent.com/wri/global-power-plant-database/master/output_database/global_power_plant_database.csv",
    low_memory=False,
)
wri_india = wri[wri["country_long"] == "India"].copy()
print(f"WRI India plants (all fuel types): {len(wri_india):,}")
print(wri_india["primary_fuel"].value_counts())

# All proximity/distance features in this notebook use the locked projected CRS.
# WRI GPPD is an external fixed dataset; no FIRMS labels or split membership are
# consulted when computing these geometric features.
h3_points_7755 = h3_gdf[["h3_lat", "h3_lon"]].copy()
h3_geom = gpd.GeoDataFrame(
    h3_points_7755,
    geometry=gpd.points_from_xy(h3_points_7755["h3_lon"], h3_points_7755["h3_lat"]),
    crs="EPSG:4326",
).to_crs(PROJECT_CRS)

def nearest_distance_km_projected(query_xy, ref_xy):
    if len(ref_xy) == 0:
        return np.full(len(query_xy), np.nan, dtype="float64")
    nn = NearestNeighbors(n_neighbors=1, algorithm="ball_tree", metric="euclidean")
    nn.fit(ref_xy)
    d, _ = nn.kneighbors(query_xy)
    return d[:, 0] / 1000.0

def count_within_radius_projected(query_xy, ref_xy, radius_km):
    if len(ref_xy) == 0:
        return np.zeros(len(query_xy), dtype="int64")
    nn = NearestNeighbors(radius=radius_km * 1000.0, algorithm="ball_tree", metric="euclidean")
    nn.fit(ref_xy)
    counts = nn.radius_neighbors(query_xy, return_distance=False)
    return np.fromiter((len(x) for x in counts), dtype="int64", count=len(query_xy))

query_xy = np.column_stack([h3_geom.geometry.x.values, h3_geom.geometry.y.values])

for fuel in wri_india["primary_fuel"].dropna().unique():
    sub = wri_india[wri_india["primary_fuel"] == fuel].copy()
    ref_geom = gpd.GeoDataFrame(
        sub,
        geometry=gpd.points_from_xy(sub["longitude"], sub["latitude"]),
        crs="EPSG:4326",
    ).to_crs(PROJECT_CRS)
    ref_xy = np.column_stack([ref_geom.geometry.x.values, ref_geom.geometry.y.values])

    col_dist = f"dist_wri_{fuel.lower().replace(' ', '_')}_km"
    col_count = f"n_wri_{fuel.lower().replace(' ', '_')}_10km"
    h3_gdf[col_dist] = nearest_distance_km_projected(query_xy, ref_xy)
    h3_gdf[col_count] = count_within_radius_projected(
        query_xy, ref_xy, radius_km=10.0
    )

print(
    "WRI feature columns added:",
    [c for c in h3_gdf.columns if "wri" in c],
)


WRI India plants (all fuel types): 1,589
primary_fuel
Solar      851
Coal       253
Hydro      233
Wind       108
Gas         68
Biomass     50
Oil         17
Nuclear      9
Name: count, dtype: int64
WRI feature columns added: ['dist_wri_solar_km', 'n_wri_solar_10km', 'dist_wri_coal_km', 'n_wri_coal_10km', 'dist_wri_wind_km', 'n_wri_wind_10km', 'dist_wri_gas_km', 'n_wri_gas_10km', 'dist_wri_hydro_km', 'n_wri_hydro_10km', 'dist_wri_biomass_km', 'n_wri_biomass_10km', 'dist_wri_oil_km', 'n_wri_oil_10km', 'dist_wri_nuclear_km', 'n_wri_nuclear_10km']


# Download nationwide OSM extract

In [7]:
import subprocess, os

pbf_path = "/kaggle/working/india-latest.osm.pbf"
if not os.path.exists(pbf_path):
    subprocess.run(["wget", "-q", "https://download.geofabrik.de/asia/india-latest.osm.pbf", "-O", pbf_path], check=True)
print(f"{pbf_path}: {os.path.getsize(pbf_path) / 1e9:.2f} GB")

/kaggle/working/india-latest.osm.pbf: 1.71 GB


# Filter OSM tags nationwide

In [8]:
PBF_FILTERED = "/kaggle/working/india_osm_filtered.osm.pbf"
GEOJSON_OUT = "/kaggle/working/india_osm_filtered.geojson"

subprocess.run([
    "osmium", "tags-filter", pbf_path,
    "nwr/landuse=industrial", "nwr/landuse=quarry", "nwr/landuse=farmland",
    "nwr/man_made=mineshaft", "nwr/man_made=adit",
    "nwr/power=plant", "nwr/power=substation", "nwr/power=generator",
    "-o", PBF_FILTERED, "--overwrite",
], check=True)

subprocess.run(["osmium", "export", PBF_FILTERED, "-o", GEOJSON_OUT, "-f", "geojson", "--overwrite"], check=True)

osm_raw = gpd.read_file(GEOJSON_OUT)
print(f"Filtered OSM features (nationwide): {len(osm_raw):,}")

Filtered OSM features (nationwide): 415,410


# OSM category distances/counts

In [9]:
CATEGORY_TAGS = {
    "industrial": ("landuse", "industrial"),
    "quarry": ("landuse", "quarry"),
    "farmland": ("landuse", "farmland"),
    "mineshaft": ("man_made", "mineshaft"),
    "adit": ("man_made", "adit"),
    "power_infra": ("power", ["plant", "substation", "generator"]),
}

# Keep the OSM features as fixed external reference data. Distances below are
# computed in the locked projected CRS. For polygon/line features the reference
# point is their EPSG:7755 centroid; this preserves the existing feature meaning
# while removing the prior CRS inconsistency.
h3_query_xy = np.column_stack([h3_gdf.geometry.centroid.x.values, h3_gdf.geometry.centroid.y.values])

# h3_gdf is already in EPSG:7755 after the state-assignment cell.
assert str(h3_gdf.crs).upper() == PROJECT_CRS.upper()

for name, (key, val) in CATEGORY_TAGS.items():
    if key not in osm_raw.columns:
        h3_gdf[f"dist_osm_{name}_km"] = np.nan
        h3_gdf[f"n_osm_{name}_5km"] = 0
        print(
            f"{name}: tag key '{key}' not present in export — "
            "0 features, check the tags-filter step"
        )
        continue

    mask = osm_raw[key].isin(val) if isinstance(val, list) else osm_raw[key] == val
    sub = osm_raw[mask].copy()
    print(f"{name}: {len(sub):,} features")

    sub_projected = sub.to_crs(PROJECT_CRS)
    rep = sub_projected.geometry.centroid
    ref_xy = np.column_stack([rep.x.values, rep.y.values])

    h3_gdf[f"dist_osm_{name}_km"] = nearest_distance_km_projected(
        h3_query_xy, ref_xy
    )
    h3_gdf[f"n_osm_{name}_5km"] = count_within_radius_projected(
        h3_query_xy, ref_xy, radius_km=5.0
    )


industrial: 54,579 features
quarry: 20,863 features
farmland: 214,181 features
mineshaft: 1,017 features
adit: 3 features
power_infra: 119,686 features


# Merge onto df_daily and save

In [10]:
h3_features = h3_gdf.drop(columns="geometry")
assert h3_features["h3_08"].is_unique
assert h3_features["state"].notna().all()
n_before = len(df_daily)

df_daily = df_daily.merge(h3_features, on="h3_08", how="left")

assert len(df_daily) == n_before, "merge changed row count — investigate before continuing"
assert df_daily["state"].notna().all(), "merge introduced rows without a state."

df_daily.to_parquet(
    "/kaggle/working/sih2026_h3_daily_features_with_osm_wri.parquet",
    index=False,
)
print(f"Saved. Final shape: {df_daily.shape}")


Saved. Final shape: (1442545, 64)


# Gas flare candidate check, cross-checked against WRI gas/oil proximity

In [11]:
flare_candidates = df_daily[
    (df_daily["is_saturated_max"] == 1) &
    (df_daily["active_days_90d"] >= 60)
].copy()

print(f"Flare candidates (saturation + persistence only): {len(flare_candidates):,} H3-days, "
      f"{flare_candidates['h3_08'].nunique():,} unique cells")
print("\n--- active_days_90d distribution among ALL saturated rows ---")
print(df_daily[df_daily["is_saturated_max"] == 1]["active_days_90d"].describe())

# Saturation + persistence alone can't tell a flare apart from a coal kiln,
# brick kiln, or large industrial furnace — anything hot and continuous
# matches this rule. Proximity to a known WRI gas/oil plant is the actual
# discriminator, and you already computed dist_wri_gas_km / dist_wri_oil_km
# earlier — use it here instead of treating the raw rule as the label.
assert "dist_wri_gas_km" in df_daily.columns, "run the WRI cell before this one"

near_wri = (
    (df_daily.loc[flare_candidates.index, "dist_wri_gas_km"] < 5) |
    (df_daily.loc[flare_candidates.index, "dist_wri_oil_km"] < 5)
)
flare_confirmed = flare_candidates[near_wri]
flare_unexplained = flare_candidates[~near_wri]

print(f"\nConfirmed (within 5km of a WRI gas/oil plant): {flare_confirmed['h3_08'].nunique():,} unique cells")
print(f"Unexplained (saturated + persistent, no nearby WRI plant — likely kilns, "
      f"furnaces, or undocumented flares, worth a manual look later): "
      f"{flare_unexplained['h3_08'].nunique():,} unique cells")

print("\n--- Threshold sensitivity: raw candidates vs WRI-confirmed, at each active_days_90d cutoff ---")
for threshold in [10, 20, 30, 45, 60, 90]:
    cands = df_daily[(df_daily["is_saturated_max"] == 1) & (df_daily["active_days_90d"] >= threshold)]
    if len(cands) == 0:
        print(f"threshold={threshold}: 0 candidates")
        continue
    near = (
        (df_daily.loc[cands.index, "dist_wri_gas_km"] < 5) |
        (df_daily.loc[cands.index, "dist_wri_oil_km"] < 5)
    )
    n_confirmed = cands.loc[near, "h3_08"].nunique()
    print(f"threshold={threshold}: {cands['h3_08'].nunique()} raw candidates, {n_confirmed} WRI-confirmed")

Flare candidates (saturation + persistence only): 423 H3-days, 35 unique cells

--- active_days_90d distribution among ALL saturated rows ---
count    54633.000000
mean         1.210825
std          7.421675
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         90.000000
Name: active_days_90d, dtype: float64

Confirmed (within 5km of a WRI gas/oil plant): 4 unique cells
Unexplained (saturated + persistent, no nearby WRI plant — likely kilns, furnaces, or undocumented flares, worth a manual look later): 31 unique cells

--- Threshold sensitivity: raw candidates vs WRI-confirmed, at each active_days_90d cutoff ---
threshold=10: 83 raw candidates, 12 WRI-confirmed
threshold=20: 60 raw candidates, 6 WRI-confirmed
threshold=30: 51 raw candidates, 6 WRI-confirmed
threshold=45: 43 raw candidates, 6 WRI-confirmed
threshold=60: 35 raw candidates, 4 WRI-confirmed
threshold=90: 4 raw candidates, 2 WRI-confirmed


# THE ACTUAL DELIVERABLE

In [12]:
def unique_cells_within(df, dist_col, threshold):
    return df.loc[df[dist_col] < threshold, "h3_08"].nunique()

EVIDENCE_RADIUS_KM = 5.0  # existing diagnostic radius; not a learned threshold

records = []
for state, g in df_daily.groupby("state", dropna=False):
    records.append({
        "state": state,
        "n_h3_days": len(g),
        "n_unique_cells": g["h3_08"].nunique(),
        "industrial_cells": unique_cells_within(
            g, "dist_osm_industrial_km", EVIDENCE_RADIUS_KM
        ),
        "quarry_cells": unique_cells_within(
            g, "dist_osm_quarry_km", EVIDENCE_RADIUS_KM
        ),
        "farmland_cells": unique_cells_within(
            g, "dist_osm_farmland_km", EVIDENCE_RADIUS_KM
        ),
        "power_infra_cells": unique_cells_within(
            g, "dist_osm_power_infra_km", EVIDENCE_RADIUS_KM
        ),
    })
state_summary = pd.DataFrame(records)

wri_cols = [
    c for c in df_daily.columns
    if c.startswith("n_wri_") and c.endswith("_10km")
]
for col in wri_cols:
    name = col.replace("n_wri_", "wri_").replace("_10km", "")
    per_state = (
        df_daily.assign(_hit=df_daily[col] > 0)
        .groupby("state", dropna=False)
        .apply(lambda g: g.loc[g["_hit"], "h3_08"].nunique())
    )
    state_summary[name] = state_summary["state"].map(per_state)

flare_by_state = (
    flare_candidates.groupby("state")["h3_08"]
    .nunique()
    .rename("flare_candidate_cells")
)
flare_confirmed_by_state = (
    flare_confirmed.groupby("state")["h3_08"]
    .nunique()
    .rename("flare_confirmed_cells")
)

state_summary = (
    state_summary
    .merge(flare_by_state, on="state", how="left")
    .merge(flare_confirmed_by_state, on="state", how="left")
)
state_summary[["flare_candidate_cells", "flare_confirmed_cells"]] = (
    state_summary[["flare_candidate_cells", "flare_confirmed_cells"]].fillna(0)
)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
print(state_summary.sort_values("n_unique_cells", ascending=False).to_string(index=False))

state_summary.to_csv(
    "/kaggle/working/state_evidence_table.csv",
    index=False,
)
print("\nSaved: /kaggle/working/state_evidence_table.csv")


/tmp/ipykernel_16/2817007304.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g["_hit"], "h3_08"].nunique())
/tmp/ipykernel_16/2817007304.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g["_hit"], "h3_08"].nunique())
/tmp/ipykernel_16/2817007304.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a fu

                                   state  n_h3_days  n_unique_cells  industrial_cells  quarry_cells  farmland_cells  power_infra_cells  wri_solar  wri_coal  wri_wind  wri_gas  wri_hydro  wri_biomass  wri_oil  wri_nuclear  flare_candidate_cells  flare_confirmed_cells
                          Madhya Pradesh     222389          123801              8858         10101            7975              23408       2040      3575       382      161       1096          157        0            0                    1.0                    0.0
                             Maharashtra     127346           84915             15493          9082           15893              23987       3052      2491      2020      727       3257         2125      794           29                    0.0                    0.0
                            Chhattisgarh     152902           69228              5329          3135            5442               9431       1226      3657         0        0         82            0 

/tmp/ipykernel_16/2817007304.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g["_hit"], "h3_08"].nunique())


In [13]:
CLASS_COLS = [
    "industrial_cells", "quarry_cells",
    "farmland_cells", "power_infra_cells", "flare_confirmed_cells",
]
MIN_EVIDENCE = 20  # diagnostic display cutoff only; never used by labeling

state_summary["classes_present"] = (
    state_summary[CLASS_COLS] >= MIN_EVIDENCE
).sum(axis=1)
ranked = (
    state_summary
    .sort_values(["classes_present", "n_h3_days"], ascending=False)
    .reset_index(drop=True)
)

display_cols = ["state", "n_h3_days", "classes_present"] + CLASS_COLS

# FIX: gmap is a multi-column DataFrame (log1p of all CLASS_COLS at once),
# not a single column. background_gradient()'s default axis=0 iterates
# column-by-column and expects gmap to match each column as a Series -
# passing a full DataFrame there raises:
#   ValueError: 'gmap' is a DataFrame but underlying data for operations
#   is a Series with 'axis in [0,1]'
# axis=None tells it to treat gmap as one full 2D array matching the whole
# subset at once, which is what a multi-column gmap actually requires.
styled = (
    ranked[display_cols]
    .style
    .background_gradient(
        subset=CLASS_COLS,
        cmap="YlOrRd",
        gmap=np.log1p(ranked[CLASS_COLS]),
        axis=None,
    )
    .background_gradient(
        subset=["classes_present"],
        cmap="Greens",
    )
    .format(precision=0)
)
styled


,state,n_h3_days,classes_present,industrial_cells,quarry_cells,farmland_cells,power_infra_cells,flare_confirmed_cells
0,Madhya Pradesh,222389,4,8858,10101,7975,23408,0
1,Chhattisgarh,152902,4,5329,3135,5442,9431,0
2,Odisha,134871,4,6564,2777,2791,9542,0
3,Maharashtra,127346,4,15493,9082,15893,23987,0
4,Uttar Pradesh,99218,4,13418,1879,7488,13673,0
5,Telangana,76853,4,7180,2175,5585,16481,0
6,Andhra Pradesh,69991,4,7590,3054,7539,11148,0
7,Karnataka,67960,4,11945,4732,6341,23836,0
8,Punjab,65916,4,16363,826,7370,24280,0
9,Jharkhand,64965,4,3378,3884,823,2122,0


In [14]:
CLASS_COLS = [
    "industrial_cells", "quarry_cells",
    "farmland_cells", "power_infra_cells", "flare_confirmed_cells",
]

def check_partition_support(
    train_states,
    testA_states,
    testB_states,
    summary,
    class_cols=CLASS_COLS,
    min_train=100,
    min_test=30,
):
    """Read-only post-freeze support diagnostic.

    This function measures counts for already-defined partitions. It does not
    choose states, alter CONFIG, fit a threshold, or feed any labeling rule.
    """
    def totals(states):
        return summary[summary["state"].isin(states)][class_cols].sum()

    train_tot = totals(train_states)
    testA_tot = totals(testA_states)
    testB_tot = totals(testB_states)

    result = {
        "train_ok": bool((train_tot >= min_train).all()),
        "testA_ok": bool((testA_tot >= min_test).all()),
        "testB_ok": bool((testB_tot >= min_test).all()),
    }
    result["valid"] = all(result.values())

    for label_name, states_list, tot in [
        ("train", train_states, train_tot),
        ("testA", testA_states, testA_tot),
        ("testB", testB_states, testB_tot),
    ]:
        print(f"\n{label_name}: {list(states_list)}")
        print(tot.to_string())

    print(f"\nVALID: {result['valid']}", result)
    return result


In [15]:
# Locked partitions only. IMPORTANT: no Test A/B support calculation occurs
# before Phase 6 rules are frozen. Support is measured in Cell 28, after labels.
assert TRAIN_STATES == [
    "Maharashtra", "Karnataka", "Madhya Pradesh",
    "Punjab", "Andhra Pradesh", "Telangana",
]
assert TEST_A_STATES == ["Gujarat", "Tamil Nadu"]
assert TEST_B_STATES == ["Jharkhand", "Rajasthan"]

print("Locked train/test partitions loaded.")


Locked train/test partitions loaded.


# PS26162 — Phase 6: Multi-class label bootstrap (4 trained classes)

Locked taxonomy: `industrial`, `mining`, `agricultural_burn`, `wildfire`.
`unclassified` is NOT a bootstrap class.

Locked rule order: `wildfire → mining → agricultural_burn → industrial`.

The fitting steps in the next code cell are TRAIN-state-only:
- wildfire FRP cutoff: 90th percentile of the TRAIN-only not-near-farmland FRP distribution;
- mining persistence: deterministic TRAIN-only persistence-decay calibration.

After fitting, both scalar values are frozen and applied to every row,
including Test A and Test B. Test A/B support is measured only after labeling.


In [16]:
"""
PS26162 — Phase 6: Multi-class label bootstrap (4 trained classes)

Locked taxonomy:
    industrial, mining, agricultural_burn, wildfire

Locked rule order:
    wildfire -> mining -> agricultural_burn -> industrial

Leakage fixes already in this cell (unchanged from prior version):
1. Wildfire FRP cutoff is fit on TRAIN-state rows only, then frozen and
   applied uniformly to TRAIN + Test A + Test B.
2. Mining persistence is recalibrated on TRAIN-state rows only.
3. Test A/Test B support is never an acceptance criterion for calibration.
4. No 'unclassified' bootstrap label is created.

CORRECTIONS made in this revision (dated, not silent):
5. BUG FIX, then SUPERSEDED BY A FINAL DECISION — mining persistence
   elbow selection used argmax(gap), which is mathematically forced to
   tie at 0 at both endpoints of any convex decreasing curve and
   therefore always selects the first candidate (day 1). Verified
   directly against the real TRAIN sweep table saved in this notebook's
   last execution: gap[day=1]=0.0, gap[day=90]=0.0, every interior day is
   negative. This produced mining_persistence_days=1 (an almost-no-op
   filter) and inflated mining support to 9,974/757/1,794
   train/TestA/TestB.
   An intermediate fix swapped to argmin(gap) (the mathematically correct
   sign), which selects day 4 (492 unique TRAIN cells). But day 4 and
   day 5 turned out to be a near-total tie (gap margin = 0.000808, ~0.09%
   of the full gap range) — statistically indistinguishable, not a real
   elbow one way or the other.
   FINAL DECISION (dated, supersedes the argmin fix): do not let a
   sign-convention pick the value when the two candidates are this close.
   mining_persistence_days is now a HARDCODED constant, not an automated
   selection — see the DECISION block below, right before
   bootstrap_labels. The full sweep and the argmin/argmax comparison are
   kept as a print-only diagnostic for defensibility (the notebook still
   shows its work), but the printed sweep no longer sets anything.
6. NAMING BUG FIX — mining's own proximity threshold was reading
   cfg["industrial_proximity_km"], silently coupling it to industrial's
   constant. Any future retune of industrial_proximity_km for industrial's
   needs would have silently moved mining's threshold too. Split into a
   separate cfg["mining_proximity_km"], same value (5.0 km) today, but
   independently scoped going forward.
7. NEW — agricultural_burn was previously proximity-only
   (dist_osm_farmland_km <= 1km, no thermal/persistence condition), which
   is the opposite of documented VIIRS/FIRMS methodology (thermal +
   persistence should be primary, proximity secondary). Added a TRAIN-fit
   FRP ceiling: near-farmland detections whose frp_max exceeds a TRAIN-only
   95th-percentile ceiling are treated as too intense to plausibly be
   routine crop/stubble burning and are NOT claimed by agri_burn — they
   fall through to industrial's checks instead, which is more likely
   correct for an intense near-farmland source that isn't a genuine
   wildfire-grade event either.
   *** THIS CEILING VALUE CANNOT BE FABRICATED. It must be produced by
   actually running this cell against df_daily on Kaggle. Do not ship a
   guessed number in its place. ***

STILL OPEN, INTENTIONALLY NOT TOUCHED IN THIS PASS (disclosed, not silently
skipped):
- Industrial's 4-of-5 OR-branches remain pure unfit proximity conditions
  (dist_osm_industrial_km, dist_osm_power_infra_km, dist_wri_gas_km,
  dist_wri_oil_km). Same structural weakness as agri_burn had, less
  exposed so far (ablation drop 0.9989->0.9477, moderate not catastrophic).
  Not fixed here — needs its own TRAIN-fit intensity/persistence condition
  per branch, deferred to a follow-up pass given the Sept 4 deadline.
- Wildfire-steals-mining structural gap (wildfire's condition doesn't
  exclude near-mining rows, so a mining blast's high-FRP first detection
  could be claimed by wildfire before the cascade reaches mining) is
  quantified in a new diagnostic cell appended after this one, but NOT
  fixed here — decide after seeing the real overlap count from Kaggle.
- TRAIN-fit-on-full-TRAIN leakage into the internal H3-hash validation
  fold (Optuna's internal signal is mildly optimistic because these
  thresholds are fit on all of TRAIN_STATES, which includes whatever later
  becomes the internal-validation subset) is a real, disclosed limitation
  of this notebook's design. It does NOT affect Test A/Test B (both fully
  outside TRAIN_STATES). Fixing it requires refitting inside the internal
  fold boundary defined in sih-catboost-training.ipynb, out of scope for
  this file alone.

RUN THIS: re-run this cell (and every cell after it, in order) on Kaggle
against the live df_daily. Every printed diagnostic below — the mining
sweep table, the wildfire cutoff, the new agri FRP ceiling, the near-tie
margin — must come from that real run before any of these numbers are
treated as frozen for submission.
"""

import pandas as pd
import numpy as np

CONFIG = {
    "frp_max": "frp_max",
    "n_detections": "n_detections",
    "is_saturated_max": "is_saturated_max",
    "active_days_90d": "active_days_90d",
    "is_first_observation": "is_first_observation",
    "is_offshore": "is_offshore",
    # NEW (2026-09-06): calendar feature from data-eda.ipynb Phase 8C.
    # Required here as a DIAGNOSTIC-ONLY input for now (see the new
    # post-freeze seasonality/persistence diagnostic cell after Cell 30) --
    # NOT wired into bootstrap_labels()'s agri_burn condition. See that
    # diagnostic cell's docstring for why a hard national month-window gate
    # was researched and deliberately NOT adopted as a label rule.
    "acq_month": "acq_month",

    "dist_osm_industrial_km": "dist_osm_industrial_km",
    "dist_osm_quarry_km": "dist_osm_quarry_km",
    "dist_osm_mineshaft_km": "dist_osm_mineshaft_km",
    "dist_osm_adit_km": "dist_osm_adit_km",
    "dist_osm_farmland_km": "dist_osm_farmland_km",
    "dist_osm_power_infra_km": "dist_osm_power_infra_km",

    "dist_wri_gas_km": "dist_wri_gas_km",
    "dist_wri_oil_km": "dist_wri_oil_km",

    # These proximity values are pre-existing rule constants, not statistically
    # fitted here. The farmland value remains explicitly unvalidated.
    "industrial_proximity_km": 5.0,
    # CORRECTION: previously mining silently reused industrial_proximity_km.
    # Same value today (5.0 km), independently scoped now — see note 6 above.
    "mining_proximity_km": 5.0,
    "farmland_proximity_km": 1.0,
    "wri_gas_oil_proximity_km": 5.0,
}

REQUIRED_LABEL_COLUMNS = [
    CONFIG["frp_max"],
    CONFIG["is_saturated_max"],
    CONFIG["active_days_90d"],
    CONFIG["is_first_observation"],
    CONFIG["is_offshore"],
    # NEW (2026-09-06): fails loud here if data-eda.ipynb's Phase 8C
    # (acq_month/doy_sin/doy_cos) has not been rerun upstream -- continuity
    # check, not a labeling-rule dependency (acq_month is not used inside
    # bootstrap_labels() itself, only in the diagnostic cell after Cell 30).
    CONFIG["acq_month"],
    CONFIG["dist_osm_industrial_km"],
    CONFIG["dist_osm_quarry_km"],
    CONFIG["dist_osm_mineshaft_km"],
    CONFIG["dist_osm_adit_km"],
    CONFIG["dist_osm_farmland_km"],
    CONFIG["dist_osm_power_infra_km"],
    CONFIG["dist_wri_gas_km"],
    CONFIG["dist_wri_oil_km"],
    "state",
    "h3_08",
]
missing = [c for c in REQUIRED_LABEL_COLUMNS if c not in df_daily.columns]
assert not missing, f"Missing required Phase 6 columns: {missing}"


def derive_wildfire_frp_cutoff(df, cfg):
    """Fit wildfire FRP cutoff on TRAIN states only, then freeze the scalar.

    UNCHANGED in this revision — already TRAIN-only, already verified
    against real saved output (froze at 11.160000 on the last real run).
    """
    fit_df = df[df["state"].isin(TRAIN_STATES)].copy()

    assert len(fit_df) > 0, "TRAIN-only wildfire fit set is empty."
    assert set(fit_df["state"].dropna().unique()).issubset(set(TRAIN_STATES))

    near_farmland = (
        fit_df[cfg["dist_osm_farmland_km"]]
        <= cfg["farmland_proximity_km"]
    )

    frp = fit_df[cfg["frp_max"]]
    not_near = frp[~near_farmland]
    near = frp[near_farmland]

    print("=" * 70)
    print("TRAIN-ONLY WILDFIRE FRP CUTOFF FIT")
    print("=" * 70)
    print(f"TRAIN fit rows: {len(fit_df):,}")
    print(f"TRAIN fit states: {sorted(fit_df['state'].dropna().unique())}")

    print("\nNOT near farmland — TRAIN ONLY:")
    print(not_near.describe(percentiles=[.5, .75, .90, .95, .99]))
    print("\nNear farmland — TRAIN ONLY:")
    print(near.describe(percentiles=[.5, .75, .90, .95, .99]))

    cutoff = not_near.quantile(0.90)
    assert pd.notna(cutoff), "TRAIN-only wildfire cutoff is NaN."

    pct_near_above_cutoff = (
        (near > cutoff).mean() * 100
        if len(near) else np.nan
    )

    print(f"\nFROZEN TRAIN-ONLY wildfire cutoff: {cutoff:.6f}")
    print(
        "% of TRAIN near-farmland detections above cutoff: "
        f"{pct_near_above_cutoff:.2f}%"
    )
    print(
        "Leakage check: cutoff fitting rows are TRAIN states only; "
        "the returned scalar is subsequently applied to every row."
    )
    return float(cutoff)


def sweep_mining_persistence_diagnostic(df, cfg):
    """TRAIN-only Phase 6c persistence sweep — DIAGNOSTIC / PRINT ONLY.

    DOES NOT SET mining_persistence_days. That constant is now hardcoded
    (see the DECISION block below this function) because the automated
    elbow methods disagree on this exact curve in a way that isn't
    resolvable by picking a "more correct" sign convention:
      - argmax(gap): mathematically broken for a convex decreasing curve
        (forced tie at 0 at both endpoints) -> always selects day 1.
        Verified against the real TRAIN sweep: gap ties at exactly 0.0 at
        both ends, negative everywhere between. This is why the old
        shipped value was 1.
      - argmin(gap): the mathematically correct sign for "max distance
        below the chord" -> selects day 4 (492 cells). But day 4 and
        day 5 are a near-total tie (margin 0.000808, ~0.09% of the gap
        range) — not a real elbow, just noise.

    This function still runs the full sweep and prints both candidates
    plus the near-tie margin, so the sweep table and the reasoning behind
    NOT trusting either automated pick remain visible and defensible in
    the notebook. It intentionally returns only the sweep DataFrame, not
    a "selected day" — to make it impossible for a future edit to
    silently wire this function's output back into CONFIG without also
    reversing this decision explicitly.

    Test A/B representation is not used anywhere in this function.
    """
    fit_df = df[df["state"].isin(TRAIN_STATES)].copy()
    assert len(fit_df) > 0, "TRAIN-only mining calibration set is empty."
    assert set(fit_df["state"].dropna().unique()).issubset(set(TRAIN_STATES))

    dist_mining = fit_df[
        [
            cfg["dist_osm_quarry_km"],
            cfg["dist_osm_mineshaft_km"],
            cfg["dist_osm_adit_km"],
        ]
    ].min(axis=1)

    # CORRECTION: was cfg["industrial_proximity_km"] — see note 6 above.
    near_mining = dist_mining <= cfg["mining_proximity_km"]

    max_persistence = int(fit_df[cfg["active_days_90d"]].max())
    candidates = range(1, max_persistence + 1)

    rows = []
    for d in candidates:
        mask = near_mining & (fit_df[cfg["active_days_90d"]] >= d)
        cand = fit_df.loc[mask].copy()
        cand_dist = dist_mining.loc[mask]

        rows.append({
            "persistence_days": int(d),
            "candidate_rows": int(mask.sum()),
            "unique_cells": int(cand["h3_08"].nunique()),
            "median_dist_km": float(cand_dist.median()) if len(cand_dist) else np.nan,
            "p90_dist_km": float(cand_dist.quantile(0.90)) if len(cand_dist) else np.nan,
            "median_frp": float(cand[cfg["frp_max"]].median()) if len(cand) else np.nan,
            "p90_frp": float(cand[cfg["frp_max"]].quantile(0.90)) if len(cand) else np.nan,
        })

    sweep = pd.DataFrame(rows)
    sweep["cell_retention_vs_prev"] = (
        sweep["unique_cells"] / sweep["unique_cells"].shift(1)
    )
    sweep["row_retention_vs_prev"] = (
        sweep["candidate_rows"] / sweep["candidate_rows"].shift(1)
    )

    # DIAGNOSTIC ONLY. Compute both the broken argmax and the corrected
    # argmin picks purely to show, in the notebook's own output, why
    # neither was trusted to set the constant. Neither value below is
    # used anywhere outside this print block.
    x = sweep["persistence_days"].to_numpy(dtype=float)
    y = sweep["unique_cells"].to_numpy(dtype=float)

    assert np.isfinite(y).all(), "Mining persistence sweep produced invalid counts."
    assert y[0] >= y[-1], "Mining candidate support must not increase with persistence."

    print("=" * 70)
    print("TRAIN-ONLY MINING PERSISTENCE SWEEP (diagnostic only)")
    print("=" * 70)
    print(f"TRAIN calibration rows: {len(fit_df):,}")
    print(f"TRAIN states used: {sorted(fit_df['state'].unique())}")
    print("\nPersistence sweep:")
    print(sweep.to_string(index=False))

    if y[0] == y[-1]:
        print("\nCurve is flat end-to-end; no elbow comparison to show.")
    else:
        x_norm = (x - x.min()) / (x.max() - x.min())
        y_norm = (y - y[-1]) / (y[0] - y[-1])
        straight = 1.0 - x_norm
        gap = y_norm - straight

        argmax_idx = int(np.argmax(gap))
        argmin_idx = int(np.argmin(gap))
        gap_range = float(gap.max() - gap.min())

        order = np.argsort(gap)  # ascending: most negative first
        best_idx, runner_idx = order[0], order[1]
        margin = float(gap[runner_idx] - gap[best_idx])
        margin_pct_of_range = (margin / gap_range * 100) if gap_range > 0 else np.nan

        print("\n--- Why neither automated pick was trusted ---")
        print(
            f"argmax(gap) [old, broken] -> day {int(x[argmax_idx])} "
            f"(gap={gap[argmax_idx]:.6f}, unique_cells={int(y[argmax_idx])}). "
            "Mathematically forced to the first day on any convex "
            "decreasing curve — endpoints tie at gap=0.0 by construction, "
            "not a real elbow."
        )
        print(
            f"argmin(gap) [sign-corrected] -> day {int(x[argmin_idx])} "
            f"(gap={gap[argmin_idx]:.6f}, unique_cells={int(y[argmin_idx])})."
        )
        print(
            f"Runner-up to argmin: day {int(x[runner_idx])} "
            f"(gap={gap[runner_idx]:.6f}, unique_cells={int(y[runner_idx])})"
        )
        print(
            f"Margin between argmin's pick and its runner-up: {margin:.6f} "
            f"({margin_pct_of_range:.3f}% of full gap range)"
        )
        if margin_pct_of_range < 1.0:
            print(
                "CONFIRMS the decision below: this margin is noise, not a "
                "real elbow. mining_persistence_days is therefore "
                "hardcoded, not auto-selected from this sweep — see the "
                "DECISION block after this function."
            )

    return sweep


def derive_agri_frp_ceiling(df, cfg):
    """TRAIN-only fit for an agricultural_burn FRP ceiling.

    NEW in this revision (correction note 7 above). The prior agri_burn
    rule was proximity-only (dist_osm_farmland_km <= 1km), which the
    project's own literature audit found to be the opposite of documented
    VIIRS/FIRMS methodology (thermal + persistence should be the primary
    discriminating signal, proximity secondary/confirmatory).

    Method: among TRAIN rows that are near farmland AND not already
    claimed by wildfire (i.e. genuine agri_burn candidates under the
    current cascade), take the 95th percentile of frp_max as a ceiling.
    Near-farmland detections above this ceiling are intense outliers
    relative to typical crop/stubble burning and are excluded from
    agri_burn — they fall through to the industrial check instead.

    This is intentionally a ceiling (upper bound), not a floor: agri_burn
    should look like the bulk of the near-farmland intensity distribution,
    not its extreme tail. The 95th-percentile choice mirrors the same
    quantile-based, TRAIN-only pattern already used for the wildfire
    cutoff (90th percentile there, applied to a different distribution/
    purpose) — same method family, so it's not an arbitrary ad hoc choice.

    *** DO NOT trust a placeholder number for this. The printed value below
    only means anything after this cell has actually been executed against
    the real df_daily on Kaggle. ***
    """
    fit_df = df[df["state"].isin(TRAIN_STATES)].copy()
    assert len(fit_df) > 0, "TRAIN-only agri ceiling fit set is empty."
    assert set(fit_df["state"].dropna().unique()).issubset(set(TRAIN_STATES))

    near_farmland = (
        fit_df[cfg["dist_osm_farmland_km"]] <= cfg["farmland_proximity_km"]
    )
    not_wildfire = ~(
        (fit_df[cfg["frp_max"]] >= WILDFIRE_FRP_CUTOFF)
        & (fit_df[cfg["is_first_observation"]] == 1)
        & (fit_df[cfg["is_offshore"]] != 1)
    )
    agri_candidates = near_farmland & not_wildfire

    frp_candidates = fit_df.loc[agri_candidates, cfg["frp_max"]]

    print("=" * 70)
    print("TRAIN-ONLY AGRICULTURAL_BURN FRP CEILING FIT")
    print("=" * 70)
    print(f"TRAIN near-farmland, non-wildfire candidate rows: {len(frp_candidates):,}")
    print(frp_candidates.describe(percentiles=[.5, .75, .90, .95, .99]))

    assert len(frp_candidates) > 0, (
        "No TRAIN-only agri_burn candidates found — cannot fit a ceiling. "
        "This would indicate a genuine data problem, not something to "
        "paper over with a fallback constant."
    )

    ceiling = float(frp_candidates.quantile(0.95))
    assert pd.notna(ceiling), "TRAIN-only agri FRP ceiling is NaN."

    pct_excluded = (frp_candidates > ceiling).mean() * 100

    print(f"\nFROZEN TRAIN-ONLY agri_burn FRP ceiling (95th pct): {ceiling:.6f}")
    print(
        f"% of TRAIN agri candidates excluded by this ceiling: {pct_excluded:.2f}% "
        "(expected ~5% by construction of the 95th-percentile choice)"
    )
    print(
        "Leakage check: ceiling fitting rows are TRAIN states only; "
        "the returned scalar is subsequently applied to every row."
    )
    return ceiling


def derive_agri_persistence_ceiling(df, cfg):
    """TRAIN-only fit for an agricultural_burn active_days_90d ceiling.

    NEW (2026-09-06). Root cause this addresses: the cell-31 post-freeze
    diagnostic (real Kaggle run, TRAIN agri_burn rows n=7,935) showed
    active_days_90d for currently-labeled agri_burn rows is >=90%
    concentrated at 0-1 days (genuinely short-lived, consistent with real
    crop/stubble burning), but the 99th percentile jumps to ~40 days and
    the max is 60 days. A 60-day "crop fire" is not physically plausible
    as a single burn event -- that tail is almost certainly contamination
    from something persistent (industrial/mining) sitting near farmland
    that the current proximity+FRP-ceiling rule is misclassifying.

    A national MONTH gate was researched and explicitly rejected (see the
    diagnostic cell after label freezing): per-state real TRAIN data shows
    Karnataka's agri_burn rows are ~80% concentrated in Jan-Mar, outside
    the literature-proposed Mar/Apr/May/Oct/Nov window, and Maharashtra
    shows a similar mismatch. A national seasonal gate would have
    destroyed real TRAIN signal for those states. The persistence ceiling
    below is a DIFFERENT, state-independent physical constraint (how long
    a single fire event plausibly persists) and is not subject to that
    same regional-mismatch problem -- short-duration is short-duration
    everywhere in India, unlike a specific calendar window.

    Method: same TRAIN-only, 95th-percentile pattern as
    derive_agri_frp_ceiling above, applied to active_days_90d instead of
    frp_max, over the same near-farmland/non-wildfire candidate rows.

    *** DO NOT trust a placeholder number for this. The printed value
    below only means anything after this cell has actually been executed
    against the real df_daily on Kaggle. ***
    """
    fit_df = df[df["state"].isin(TRAIN_STATES)].copy()
    assert len(fit_df) > 0, "TRAIN-only agri persistence ceiling fit set is empty."
    assert set(fit_df["state"].dropna().unique()).issubset(set(TRAIN_STATES))

    near_farmland = (
        fit_df[cfg["dist_osm_farmland_km"]] <= cfg["farmland_proximity_km"]
    )
    not_wildfire = ~(
        (fit_df[cfg["frp_max"]] >= WILDFIRE_FRP_CUTOFF)
        & (fit_df[cfg["is_first_observation"]] == 1)
        & (fit_df[cfg["is_offshore"]] != 1)
    )
    agri_candidates = near_farmland & not_wildfire

    persistence_candidates = fit_df.loc[agri_candidates, cfg["active_days_90d"]]

    print("=" * 70)
    print("TRAIN-ONLY AGRICULTURAL_BURN PERSISTENCE CEILING FIT")
    print("=" * 70)
    print(f"TRAIN near-farmland, non-wildfire candidate rows: {len(persistence_candidates):,}")
    print(persistence_candidates.describe(percentiles=[.5, .75, .90, .95, .99]))

    assert len(persistence_candidates) > 0, (
        "No TRAIN-only agri_burn candidates found — cannot fit a "
        "persistence ceiling. This would indicate a genuine data "
        "problem, not something to paper over with a fallback constant."
    )

    ceiling = float(persistence_candidates.quantile(0.95))
    assert pd.notna(ceiling), "TRAIN-only agri persistence ceiling is NaN."

    pct_excluded = (persistence_candidates > ceiling).mean() * 100

    print(f"\nFROZEN TRAIN-ONLY agri_burn persistence ceiling (95th pct): {ceiling:.6f} days")
    print(
        f"% of TRAIN agri candidates excluded by this ceiling: {pct_excluded:.2f}% "
        "(expected ~5% by construction of the 95th-percentile choice)"
    )
    print(
        "Leakage check: ceiling fitting rows are TRAIN states only; "
        "the returned scalar is subsequently applied to every row."
    )
    return ceiling


# Runs the sweep and prints the argmax-vs-argmin comparison for the record.
# Returns the sweep table only — does NOT set mining_persistence_days.
MINING_CALIBRATION_SWEEP = sweep_mining_persistence_diagnostic(df_daily, CONFIG)

# ------------------------------------------------------------------------
# DECISION (final, dated — supersedes both the argmax bug and the argmin
# fix). Automated elbow detection found day 4 and day 5 statistically tied
# (margin 0.000808, ~0.09% of the gap range — see sweep diagnostic above).
# No defensible methodology distinguishes them. Rather than let a sign
# convention make the call, mining_persistence_days is hardcoded to the
# team's prior accepted reference value.
# ------------------------------------------------------------------------
MINING_PERSISTENCE_DAYS = 5  # Automated elbow detection found day 4/5
# statistically tied (Δ<0.1% of range); day 5 retained as it matches prior
# accepted value pending full recalibration post-submission.

WILDFIRE_FRP_CUTOFF = derive_wildfire_frp_cutoff(df_daily, CONFIG)

AGRI_FRP_CEILING = derive_agri_frp_ceiling(df_daily, CONFIG)

# NEW (2026-09-06): TRAIN-fit persistence ceiling for agricultural_burn.
# Same discipline as the other three frozen constants above -- fit on
# TRAIN states only, derived from the real cell-31 diagnostic finding
# (99th pct ~40 days / max 60 days is not physically plausible for a
# single crop-burn event). See derive_agri_persistence_ceiling's
# docstring for the full rationale and why a persistence ceiling was
# adopted while a national month/season gate was explicitly rejected.
AGRI_PERSISTENCE_CEILING = derive_agri_persistence_ceiling(df_daily, CONFIG)

# Freeze the values before any label application. Do not overwrite any of
# these four scalars using Test A/B diagnostics later.
CONFIG["mining_persistence_days"] = MINING_PERSISTENCE_DAYS
CONFIG["wildfire_frp_cutoff"] = WILDFIRE_FRP_CUTOFF
CONFIG["agri_frp_ceiling"] = AGRI_FRP_CEILING
CONFIG["agri_persistence_ceiling"] = AGRI_PERSISTENCE_CEILING


def bootstrap_labels(df, cfg):
    """Apply the frozen Phase 6 rule cascade uniformly to all rows."""
    df = df.copy()

    # The frozen fitting statistics are attached to CONFIG; application is
    # intentionally nationwide, including blind holdouts.
    assert pd.notna(cfg["wildfire_frp_cutoff"])
    assert int(cfg["mining_persistence_days"]) >= 1
    assert pd.notna(cfg["agri_frp_ceiling"])
    assert pd.notna(cfg["agri_persistence_ceiling"])

    label = pd.Series(pd.NA, index=df.index, dtype="object")

    # ------------------------------------------------------------------
    # 1. WILDFIRE
    #    STILL OPEN: this condition does not exclude near-mining rows, so
    #    a real mining blast's high-FRP first detection could be claimed
    #    here before the cascade reaches the mining rule. Quantified in
    #    the diagnostic cell appended after this one — not fixed yet,
    #    pending the real overlap count.
    # ------------------------------------------------------------------
    is_wildfire = (
        (df[cfg["frp_max"]] >= cfg["wildfire_frp_cutoff"])
        & (df[cfg["is_first_observation"]] == 1)
        & (df[cfg["is_offshore"]] != 1)
    )
    label[is_wildfire] = "wildfire"

    # ------------------------------------------------------------------
    # 2. MINING
    #    CORRECTION: distance threshold now uses its own
    #    mining_proximity_km instead of silently reusing
    #    industrial_proximity_km. Persistence threshold now comes from
    #    the hardcoded decision value (mining_persistence_days = 5) — see
    #    the DECISION block near sweep_mining_persistence_diagnostic's
    #    call site for the full rationale.
    # ------------------------------------------------------------------
    dist_mining = df[
        [
            cfg["dist_osm_quarry_km"],
            cfg["dist_osm_mineshaft_km"],
            cfg["dist_osm_adit_km"],
        ]
    ].min(axis=1)

    # mining_persistence_days = 5 is a hardcoded decision (not an
    # automated elbow pick) — see the DECISION block above
    # sweep_mining_persistence_diagnostic's call site for the full
    # rationale (day 4/day 5 statistical tie, day 5 matches prior
    # accepted reference value).
    is_mining = (
        label.isna()
        & (dist_mining <= cfg["mining_proximity_km"])
        & (
            df[cfg["active_days_90d"]]
            >= cfg["mining_persistence_days"]
        )
    )
    label[is_mining] = "mining"

    # ------------------------------------------------------------------
    # 3. AGRICULTURAL_BURN
    #    CORRECTION 1: TRAIN-fit FRP ceiling on top of the existing
    #    proximity condition. Previously proximity-only; now requires
    #    near-farmland AND not an intensity outlier relative to typical
    #    TRAIN agri candidates.
    #    CORRECTION 2 (2026-09-06): TRAIN-fit persistence CEILING added.
    #    Real cell-31 diagnostic (TRAIN agri_burn rows, n=7,935) showed
    #    >=90% of currently-labeled rows are 0-1 active days (consistent
    #    with genuine short-lived crop burns), but a 99th-pct tail out to
    #    ~40-60 days is not physically plausible as a single burn event --
    #    almost certainly persistent industrial/mining contamination
    #    sitting near farmland. This excludes that tail without touching
    #    the >=90% of rows that already look like real crop fires.
    #    A national MONTH/season gate was researched and explicitly
    #    REJECTED here -- real per-state TRAIN data (same diagnostic cell)
    #    showed Karnataka's agri_burn rows are ~80% concentrated in
    #    Jan-Mar and Maharashtra similarly mismatched against the
    #    literature-proposed Mar/Apr/May/Oct/Nov window. A national
    #    seasonal gate would have destroyed real TRAIN signal for those
    #    states, so it is NOT part of this rule. The persistence ceiling
    #    is a physical constraint independent of region/crop calendar,
    #    so it does not carry that same regional-mismatch risk.
    # ------------------------------------------------------------------
    near_farmland = (
        df[cfg["dist_osm_farmland_km"]] <= cfg["farmland_proximity_km"]
    )
    is_agri = (
        label.isna()
        & near_farmland
        & (df[cfg["frp_max"]] <= cfg["agri_frp_ceiling"])
        & (df[cfg["active_days_90d"]] <= cfg["agri_persistence_ceiling"])
    )
    label[is_agri] = "agricultural_burn"

    # ------------------------------------------------------------------
    # 4. INDUSTRIAL
    #    STILL OPEN: 4 of 5 OR-branches remain pure unfit proximity
    #    conditions (dist_osm_industrial_km, dist_osm_power_infra_km,
    #    dist_wri_gas_km, dist_wri_oil_km). Not fixed in this pass — see
    #    docstring at the top of this cell for why this is deliberately
    #    deferred, not overlooked.
    # ------------------------------------------------------------------
    is_industrial = (
        label.isna()
        & (
            (
                (df[cfg["is_saturated_max"]] == 1)
                & (df[cfg["active_days_90d"]] >= 60)
            )
            | (
                df[cfg["dist_osm_industrial_km"]]
                <= cfg["industrial_proximity_km"]
            )
            | (
                df[cfg["dist_osm_power_infra_km"]]
                <= cfg["industrial_proximity_km"]
            )
            | (
                df[cfg["dist_wri_gas_km"]]
                <= cfg["wri_gas_oil_proximity_km"]
            )
            | (
                df[cfg["dist_wri_oil_km"]]
                <= cfg["wri_gas_oil_proximity_km"]
            )
        )
    )
    label[is_industrial] = "industrial"

    df["label"] = label

    # This intentionally overwrites/supersedes the upstream data-eda meaning
    # of is_labeled in the OUTPUT TABLE ONLY. df_daily retains its upstream
    # diagnostic flag; df_labeled is the Phase-6 ground-truth artifact.
    df["is_labeled"] = label.notna().astype("int8")

    assert set(df.loc[df["is_labeled"] == 1, "label"].dropna().unique()).issubset(
        {"industrial", "mining", "agricultural_burn", "wildfire"}
    )

    print("\n" + "=" * 70)
    print("LABEL DISTRIBUTION (4 TRAINED CLASSES)")
    print("=" * 70)
    print(df["label"].value_counts(dropna=False))

    unlabeled = (df["is_labeled"] == 0).sum()
    print(
        f"\nUnlabeled rows (excluded from training, NOT 'unclassified'): "
        f"{unlabeled:,} ({unlabeled / len(df) * 100:.2f}%)"
    )

    print("\n" + "=" * 70)
    print("FROZEN FIT STATISTICS")
    print("=" * 70)
    print(f"Wildfire FRP cutoff (TRAIN-only): {cfg['wildfire_frp_cutoff']:.6f}")
    print(f"Mining persistence days (TRAIN-only): {cfg['mining_persistence_days']}")
    print(f"Agri_burn FRP ceiling (TRAIN-only): {cfg['agri_frp_ceiling']:.6f}")
    print(f"Agri_burn persistence ceiling (TRAIN-only): {cfg['agri_persistence_ceiling']:.6f} days")

    return df


TRAIN-ONLY MINING PERSISTENCE SWEEP (diagnostic only)
TRAIN calibration rows: 630,455
TRAIN states used: ['Andhra Pradesh', 'Karnataka', 'Madhya Pradesh', 'Maharashtra', 'Punjab', 'Telangana']

Persistence sweep:
 persistence_days  candidate_rows  unique_cells  median_dist_km  p90_dist_km  median_frp  p90_frp  cell_retention_vs_prev  row_retention_vs_prev
                1           39311          9974        2.309414     4.354844       1.820    7.370                     NaN                    NaN
                2           26545          2537        1.941496     3.905836       1.440    4.520                0.254361               0.675256
                3           23433           850        1.805929     3.734950       1.380    3.620                0.335041               0.882765
                4           22244           492        1.782695     3.711465       1.370    3.450                0.578824               0.949260
                5           21443           388        1.78269

In [17]:
# Apply the already-frozen Phase 6 rules to EVERY row.
df_labeled = bootstrap_labels(df_daily, CONFIG)

assert len(df_labeled) == len(df_daily)
assert "label" in df_labeled.columns
assert "is_labeled" in df_labeled.columns
# Frozen parameters are learned before this call; application is nationwide.
assert CONFIG["wildfire_frp_cutoff"] == WILDFIRE_FRP_CUTOFF
assert CONFIG["mining_persistence_days"] == MINING_PERSISTENCE_DAYS
# CORRECTION: new frozen constant added in this revision — assert it froze
# correctly the same way the other two already did.
assert CONFIG["agri_frp_ceiling"] == AGRI_FRP_CEILING



LABEL DISTRIBUTION (4 TRAINED CLASSES)
label
<NA>                 784705
industrial           450553
mining               108478
wildfire              82968
agricultural_burn     15841
Name: count, dtype: int64

Unlabeled rows (excluded from training, NOT 'unclassified'): 784,705 (54.40%)

FROZEN FIT STATISTICS
Wildfire FRP cutoff (TRAIN-only): 11.160000
Mining persistence days (TRAIN-only): 5
Agri_burn FRP ceiling (TRAIN-only): 10.330000
Agri_burn persistence ceiling (TRAIN-only): 27.350000 days


In [18]:
df_labeled["h3_08"] = df_labeled["h3_08"].astype(str)

label_state_counts = (
    df_labeled[df_labeled["is_labeled"] == 1]
    .groupby(["state", "label"])["h3_08"]
    .nunique()
    .unstack(fill_value=0)
    .reset_index()
)

label_cols = [
    "industrial", "mining",
    "agricultural_burn", "wildfire",
]

# POST-FREEZE ONLY: this measures support after all rule parameters are frozen
# and applied. It does not feed back into any threshold or rule decision.
support_result = check_partition_support(
    train_states=TRAIN_STATES,
    testA_states=TEST_A_STATES,
    testB_states=TEST_B_STATES,
    summary=label_state_counts,
    class_cols=label_cols,
)

print("\nPost-freeze support diagnostic:", support_result)



train: ['Maharashtra', 'Karnataka', 'Madhya Pradesh', 'Punjab', 'Andhra Pradesh', 'Telangana']
label
industrial           129932
mining                  388
agricultural_burn      5289
wildfire              37962

testA: ['Gujarat', 'Tamil Nadu']
label
industrial           23106
mining                  47
agricultural_burn      647
wildfire              3562

testB: ['Jharkhand', 'Rajasthan']
label
industrial           10832
mining                 342
agricultural_burn      119
wildfire              3766

VALID: True {'train_ok': True, 'testA_ok': True, 'testB_ok': True, 'valid': True}

Post-freeze support diagnostic: {'train_ok': True, 'testA_ok': True, 'testB_ok': True, 'valid': True}


In [19]:
# POST-FREEZE DIAGNOSTIC ONLY.
# This reproduces the "industrial would-have-stolen agri" sanity check using
# the actual frozen TRAIN-derived wildfire cutoff from the frozen TRAIN-only fit.
near_farmland = (
    df_daily[CONFIG["dist_osm_farmland_km"]]
    <= CONFIG["farmland_proximity_km"]
)
not_wildfire = ~(
    (df_daily[CONFIG["frp_max"]] >= CONFIG["wildfire_frp_cutoff"])
    & (df_daily[CONFIG["is_first_observation"]] == 1)
    & (df_daily[CONFIG["is_offshore"]] != 1)
)
candidate_agri = near_farmland & not_wildfire

would_be_industrial = candidate_agri & (
    (
        (df_daily[CONFIG["is_saturated_max"]] == 1)
        & (df_daily[CONFIG["active_days_90d"]] >= 60)
    )
    | (
        df_daily[CONFIG["dist_osm_industrial_km"]]
        <= CONFIG["industrial_proximity_km"]
    )
    | (
        df_daily[CONFIG["dist_osm_power_infra_km"]]
        <= CONFIG["industrial_proximity_km"]
    )
    | (
        df_daily[CONFIG["dist_wri_gas_km"]]
        <= CONFIG["wri_gas_oil_proximity_km"]
    )
    | (
        df_daily[CONFIG["dist_wri_oil_km"]]
        <= CONFIG["wri_gas_oil_proximity_km"]
    )
)

print(f"Frozen wildfire cutoff used: {CONFIG['wildfire_frp_cutoff']:.6f}")
print(f"Near-farmland candidates: {candidate_agri.sum():,}")
print(
    "...claimed by industrial before agri check: "
    f"{would_be_industrial.sum():,} "
    f"({would_be_industrial.sum() / max(1, candidate_agri.sum()) * 100:.1f}% of near-farmland candidates)"
)


Frozen wildfire cutoff used: 11.160000
Near-farmland candidates: 18,241
...claimed by industrial before agri check: 11,415 (62.6% of near-farmland candidates)


In [20]:
# POST-FREEZE DIAGNOSTIC ONLY — NEW in this revision.
#
# Quantifies the previously-unquantified "wildfire steals mining" structural
# gap: wildfire's condition (frp_max >= cutoff AND is_first_observation AND
# not offshore) does not exclude rows that are also near a quarry/mineshaft/
# adit. Because wildfire fires FIRST in the cascade, a real mining site's
# high-FRP opening blast could be claimed as wildfire before the cascade
# ever reaches the mining rule.
#
# This cell measures how often that's actually possible under the frozen
# thresholds. It does NOT change any label — it is read-only, exactly like
# the existing cell above that measured "industrial steals from agri".
#
# Decide what to do with this number after seeing it:
#   - If the overlap is small (a handful of rows), it may be an acceptable,
#     disclosed limitation for Sept 4.
#   - If it's non-trivial, the fix is to add a mining-proximity exclusion to
#     the wildfire condition, mirroring how the agri/industrial ordering
#     was already fixed to prevent a similar steal.

dist_mining_all = df_daily[
    [
        CONFIG["dist_osm_quarry_km"],
        CONFIG["dist_osm_mineshaft_km"],
        CONFIG["dist_osm_adit_km"],
    ]
].min(axis=1)
near_mining_all = dist_mining_all <= CONFIG["mining_proximity_km"]

is_wildfire_condition = (
    (df_daily[CONFIG["frp_max"]] >= CONFIG["wildfire_frp_cutoff"])
    & (df_daily[CONFIG["is_first_observation"]] == 1)
    & (df_daily[CONFIG["is_offshore"]] != 1)
)

wildfire_near_mining = is_wildfire_condition & near_mining_all

print(f"Frozen wildfire cutoff used: {CONFIG['wildfire_frp_cutoff']:.6f}")
print(f"Frozen mining proximity used: {CONFIG['mining_proximity_km']} km")
print(f"Rows satisfying the wildfire condition: {is_wildfire_condition.sum():,}")
print(
    "...of those, also within mining_proximity_km of a quarry/mineshaft/adit: "
    f"{wildfire_near_mining.sum():,} "
    f"({wildfire_near_mining.sum() / max(1, is_wildfire_condition.sum()) * 100:.2f}% "
    "of wildfire-condition rows)"
)
print(
    f"Unique H3 cells affected: {df_daily.loc[wildfire_near_mining, 'h3_08'].nunique():,}"
)
print(
    "\nThese rows are currently labeled 'wildfire' by rule order, not "
    "'mining' — this print does not change that. Inspect the count above "
    "before deciding whether this needs a rule-order fix before submission."
)


Frozen wildfire cutoff used: 11.160000
Frozen mining proximity used: 5.0 km
Rows satisfying the wildfire condition: 82,968
...of those, also within mining_proximity_km of a quarry/mineshaft/adit: 4,680 (5.64% of wildfire-condition rows)
Unique H3 cells affected: 4,680

These rows are currently labeled 'wildfire' by rule order, not 'mining' — this print does not change that. Inspect the count above before deciding whether this needs a rule-order fix before submission.


In [21]:
# POST-FREEZE DIAGNOSTIC ONLY -- NEW (2026-09-06). Does NOT change
# df_labeled["label"], does NOT create a new trained class, does NOT touch
# CONFIG or bootstrap_labels(). Same status as the two diagnostics above.
#
# WHY THIS EXISTS: a prior research pass proposed hard-gating agri_burn on
# acq_month.isin([3,4,5,10,11]) (India's well-documented Mar-May/Oct-Nov
# crop-residue-burning windows) as a REQUIRED condition in bootstrap_labels().
#
# THAT PROPOSAL WAS CHECKED AGAINST REAL LITERATURE (2026-09-06) AND
# REJECTED FOR THE RULE LAYER. Findings:
#   - The Mar-May / Oct-Nov windows are real and well-documented, but
#     specifically for Indo-Gangetic Plain wheat-rice rotation states
#     (Punjab, Haryana, UP, Bihar) -- e.g. NASA Earth Observatory's "A Busy
#     Season for Crop Fires in Northwestern India" and the wheat-residue
#     literature both frame this as a Punjab/Haryana/UP/MP phenomenon, not
#     a national one.
#   - Tamil Nadu -- one of our two Test A (blind) states -- runs on THREE
#     distinct cropping-season harvests (Samba: Jan-Feb, Navarai: Feb-Mar,
#     Sornavari: Aug-Sep), none of which line up with the Mar-May/Oct-Nov
#     window at all.
#   - TRAIN_STATES mix Indo-Gangetic-adjacent states (Punjab, Madhya
#     Pradesh) with peninsular states (Maharashtra, Karnataka, Andhra
#     Pradesh, Telangana) that grow rice/sugarcane/cotton on their own
#     calendars -- a single hardcoded national window would not even be
#     internally consistent across TRAIN, let alone Test A/B.
#   - A specific "~78% of crop fires are under 1 hectare" figure floated in
#     that same research pass could NOT be verified against any source
#     found -- treat it as unconfirmed, do not cite it to judges.
#
# CONCLUSION: hard-gating the LABEL RULE on a national month window would
# risk introducing a NEW pseudo-labeling bias -- systematically excluding
# genuine peninsular-state agri_burn events from the very Test A state
# (Tamil Nadu) this project is scored on for blind generalization. That is
# worse than the status quo. acq_month/doy_sin/doy_cos are therefore kept
# as MODEL features only (already wired via data-eda.ipynb Phase 8C into
# sih-catboost-training-resume.ipynb's feature manifest) -- CatBoost can
# learn a state-conditional seasonal pattern from data directly, without
# the label rule baking in a wrong assumption for 4+ of our 10 locked
# states.
#
# This cell instead does what the sweep_mining_persistence_diagnostic /
# derive_agri_frp_ceiling functions above already do: show the real
# TRAIN-only distribution before anyone decides whether a rule change is
# warranted. Two things are checked, per TRAIN state:
#   1. Month distribution of current agri_burn-labeled rows (does the
#      existing proximity+ceiling rule already cluster in expected months
#      per state, without being told to?)
#   2. active_days_90d distribution of current agri_burn-labeled rows
#      (the previously-proposed "persistence ceiling, mirroring but
#      inverting mining's floor" -- also NOT implemented below, diagnostic
#      only, so a real ceiling isn't guessed the same way the FRP ceiling
#      wasn't guessed).

agri_train = df_labeled[
    (df_labeled["label"] == "agricultural_burn")
    & (df_labeled["state"].isin(TRAIN_STATES))
].copy()

print("=" * 70)
print("AGRI_BURN SEASONALITY + PERSISTENCE DIAGNOSTIC (TRAIN-only, print-only)")
print("=" * 70)
print(f"TRAIN agri_burn rows: {len(agri_train):,}")

print("\n--- Month distribution by state (rows) ---")
month_by_state = (
    agri_train
    .groupby(["state", CONFIG["acq_month"]])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=range(1, 13), fill_value=0)
)
print(month_by_state.to_string())

print(
    "\nNOTE: if any TRAIN state's agri_burn rows cluster OUTSIDE "
    "Mar/Apr/May/Oct/Nov, that is direct evidence against a national "
    "month gate for THIS partition specifically -- inspect before ever "
    "reconsidering the rejected proposal above."
)

print("\n--- active_days_90d distribution (TRAIN agri_burn rows) ---")
print(
    agri_train[CONFIG["active_days_90d"]]
    .describe(percentiles=[.5, .75, .90, .95, .99])
)

print(
    "\nThis cell sets no config and changes no label. If a persistence "
    "ceiling or a region-aware (not national) seasonal condition is "
    "adopted later, it must be derived from this real distribution the "
    "same way wildfire_frp_cutoff and agri_frp_ceiling were -- not "
    "guessed -- and re-verified against Test A (Tamil Nadu especially) "
    "before being treated as frozen."
)


AGRI_BURN SEASONALITY + PERSISTENCE DIAGNOSTIC (TRAIN-only, print-only)
TRAIN agri_burn rows: 7,830

--- Month distribution by state (rows) ---
acq_month        1    2    3    4    5   6   7   8   9    10   11   12
state                                                                 
Andhra Pradesh   75  110  179  317  451  60   5   8   9   24  107  647
Karnataka       246  474  361   76   21   9   9   1   2    7   41  113
Madhya Pradesh   10   24  246  374   46   8   1   8   6   27  124   29
Maharashtra     236  309  529  243  105  22   0   1   3    9   99  224
Punjab           15   30   24   52  564  76  22   0  17  167  217   16
Telangana        40   55  106   41  139  19   0   0   0    5  126   64

NOTE: if any TRAIN state's agri_burn rows cluster OUTSIDE Mar/Apr/May/Oct/Nov, that is direct evidence against a national month gate for THIS partition specifically -- inspect before ever reconsidering the rejected proposal above.

--- active_days_90d distribution (TRAIN agri_burn rows)

In [22]:
# CatBoost-ready export contract.
# These source/provenance fields must not enter the model matrix:
# - is_static_land originates from the archive-only FIRMS fire_type field;
# - is_offshore participates directly in the wildfire labeling rule;
# - state-assignment diagnostics describe the join-resolution path, not a
#   thermal-source feature.
# The fields remain available in df_daily while labels are computed above, but
# are deliberately removed from the persisted training artifact.
CATBOOST_EXPORT_EXCLUDE = [
    "is_static_land",
    "is_offshore",
    "state_assignment_method",
    "_state_distance_km",
]

missing_export_columns = [
    column for column in CATBOOST_EXPORT_EXCLUDE
    if column not in df_labeled.columns
]
assert not missing_export_columns, (
    "Expected source/provenance columns are missing before export: "
    f"{missing_export_columns}"
)

df_labeled = df_labeled.drop(columns=CATBOOST_EXPORT_EXCLUDE)

# Keep the two documented CatBoost categorical fields in a stable string form.
for categorical_column in ["h3_08", "daynight"]:
    assert df_labeled[categorical_column].notna().all(), (
        f"{categorical_column} contains missing values after labeling."
    )
    df_labeled[categorical_column] = (
        df_labeled[categorical_column].astype("string").astype(str)
    )

assert not df_labeled.duplicated(["h3_08", "acq_date"]).any(), (
    "The labeled artifact contains duplicate H3-day rows."
)
expected_is_labeled = df_labeled["label"].notna().astype("int8")
assert df_labeled["is_labeled"].eq(expected_is_labeled).all(), (
    "is_labeled must exactly equal label.notna() in the Phase 6 artifact."
)
assert set(df_labeled.loc[expected_is_labeled.eq(1), "label"].unique()) <= {
    "industrial", "mining", "agricultural_burn", "wildfire",
}
assert df_labeled.loc[expected_is_labeled.eq(0), "label"].isna().all()

# The downstream CatBoost notebook excludes state/acq_date and treats only
# h3_08/daynight as categorical. Fail before export if another string-like
# feature would otherwise enter the numeric model matrix.
string_like = set(
    df_labeled.select_dtypes(include=["object", "string", "category"]).columns
)
allowed_string_like = {"h3_08", "daynight", "state", "label"}
unexpected_string_like = sorted(string_like - allowed_string_like)
assert not unexpected_string_like, (
    "Unexpected string-like columns would break the CatBoost feature matrix: "
    f"{unexpected_string_like}"
)

print("CatBoost-ready export contract passed.")
print("Exported categorical columns: ['h3_08', 'daynight']")
print(f"Excluded source/provenance columns: {CATBOOST_EXPORT_EXCLUDE}")


CatBoost-ready export contract passed.
Exported categorical columns: ['h3_08', 'daynight']
Excluded source/provenance columns: ['is_static_land', 'is_offshore', 'state_assignment_method', '_state_distance_km']


In [23]:
assert df_labeled["is_labeled"].isin([0, 1]).all()
assert set(
    df_labeled.loc[df_labeled["is_labeled"] == 1, "label"].dropna().unique()
).issubset({"industrial", "mining", "agricultural_burn", "wildfire"})

df_labeled.to_parquet(
    "/kaggle/working/sih2026_h3_daily_labeled.parquet",
    index=False,
)
print(f"Saved. Shape: {df_labeled.shape}")
print(
    f"FINAL TRAIN-ONLY wildfire FRP cutoff: "
    f"{CONFIG['wildfire_frp_cutoff']:.6f}"
)
print(
    f"FINAL TRAIN-ONLY mining_persistence_days: "
    f"{CONFIG['mining_persistence_days']}"
)
print(
    f"FINAL TRAIN-ONLY agri_frp_ceiling: "
    f"{CONFIG['agri_frp_ceiling']:.6f}"
)
print(
    f"FINAL TRAIN-ONLY agri_persistence_ceiling: "
    f"{CONFIG['agri_persistence_ceiling']:.6f} days"
)


Saved. Shape: (1442545, 61)
FINAL TRAIN-ONLY wildfire FRP cutoff: 11.160000
FINAL TRAIN-ONLY mining_persistence_days: 5
FINAL TRAIN-ONLY agri_frp_ceiling: 10.330000
FINAL TRAIN-ONLY agri_persistence_ceiling: 27.350000 days


In [24]:
# POST-FREEZE SANITY DIAGNOSTIC ONLY.
# This is descriptive, not a threshold fit and not used to change labels.

mining_sample = df_labeled[
    df_labeled["label"] == "mining"
].sample(
    n=min(15, (df_labeled["label"] == "mining").sum()),
    random_state=42,
)

cols_to_check = [
    "h3_08", "state", "h3_lat", "h3_lon",
    "dist_osm_quarry_km",
    "dist_osm_mineshaft_km",
    "dist_osm_adit_km",
    "active_days_90d",
    "frp_max",
    "n_detections",
]
print(mining_sample[cols_to_check].to_string(index=False))

print("\n--- Full mining-class distance distribution (diagnostic only) ---")
dist_mining_all = df_labeled.loc[
    df_labeled["label"] == "mining",
    [
        "dist_osm_quarry_km",
        "dist_osm_mineshaft_km",
        "dist_osm_adit_km",
    ],
].min(axis=1)
print(dist_mining_all.describe())

print("\n--- Mining rows by state (top 10; diagnostic only) ---")
print(
    df_labeled[df_labeled["label"] == "mining"]["state"]
    .value_counts()
    .head(10)
)


          h3_08          state    h3_lat    h3_lon  dist_osm_quarry_km  dist_osm_mineshaft_km  dist_osm_adit_km  active_days_90d  frp_max  n_detections
883ca97541fffff    West Bengal 23.690616 87.127465            3.507027             173.857347       1601.887254               29     1.46             1
883ca9c867fffff      Jharkhand 23.769020 86.406410            0.730878             235.703573       1543.836720               86     3.89             5
883c848ec1fffff   Chhattisgarh 22.085329 83.119435            1.599139             426.998508       1419.793345               52     2.54             3
883ca548d9fffff Madhya Pradesh 24.137967 82.637622            1.811535             598.706094       1264.601252               27     2.47             1
88618c6243fffff     Tamil Nadu 12.854419 79.907868            1.126532             178.318058        382.616218               16     1.09             1
883caa4d55fffff         Odisha 21.990704 85.326562            1.583309             313.2

In [25]:
# POST-FREEZE DIAGNOSTIC ONLY — NEW. Does NOT change df_labeled["label"],
# does NOT create a new trained class, does NOT touch CONFIG or
# bootstrap_labels(). This is a reporting sub-flag on rows already labeled
# "industrial", for judge Q&A ("how do you handle gas flares specifically?")
# — not a modeling change.
#
# WHY THIS EXISTS: the project intentionally retired gas flare as a
# standalone trained class (see cell 26 CONFIG comments — its distance
# features are folded into industrial's rule as OR-branches, not a
# separate label). This flag demonstrates that a flare-like sub-population
# is still identifiable and inspectable within the industrial class,
# without re-opening the taxonomy this close to submission.
#
# WHAT THIS IS BUILT FROM, AND WHAT IT ISN'T:
# - "high 90-day persistence", "high FRP", "industrial land-use" are all
#   real columns already in df_daily: active_days_90d, frp_max,
#   dist_osm_industrial_km, dist_wri_gas_km, dist_wri_oil_km.
# - "zero canopy" (from the unverified "6 NTRO Target Classes" text) is
#   NOT implemented. There is no canopy/forest-cover/NDVI/land-cover
#   column anywhere in this notebook (checked directly, zero matches) —
#   faking a proxy for a feature that doesn't exist would be exactly the
#   kind of fabricated-looking rigor this project has already been burned
#   by. It's just dropped, not substituted.
# - The 75-day persistence threshold is a HEURISTIC taken from that same
#   unverified text, NOT a TRAIN-only fit like wildfire_frp_cutoff,
#   mining_persistence_days, or agri_frp_ceiling. Disclose it as such if
#   this comes up with judges — it's a reasonable illustrative cut, not a
#   calibrated constant.

GAS_FLARE_PERSISTENCE_HEURISTIC_DAYS = 75  # heuristic, NOT TRAIN-fit — see note above

# DEFENSIVE CHECK — do not remove. near_gas_oil/high_persistence below are
# computed off df_daily while is_industrial_row is computed off
# df_labeled["label"]. This is only safe because df_labeled's index and row
# order are identical to df_daily's (bootstrap_labels() starts with
# df.copy(), and every intermediate cell only adds/drops columns or builds
# separate derived variables — traced cell-by-cell, never reassigns
# df_labeled via a sort/filter/reset_index). If any future edit to an
# earlier cell breaks that, this assertion fails loudly here instead of
# silently misaligning the printed counts below.
assert df_labeled.index.equals(df_daily.index), (
    "df_labeled and df_daily indices have diverged — the boolean masks "
    "below (built from df_daily) would misalign against df_labeled's "
    "label column. Fix the upstream cell that reindexed/filtered/sorted "
    "df_labeled before trusting any count printed by this cell."
)

is_industrial_row = df_labeled["label"] == "industrial"

near_gas_oil = (
    (df_daily[CONFIG["dist_wri_gas_km"]] <= CONFIG["wri_gas_oil_proximity_km"])
    | (df_daily[CONFIG["dist_wri_oil_km"]] <= CONFIG["wri_gas_oil_proximity_km"])
)

high_persistence = df_daily[CONFIG["active_days_90d"]] >= GAS_FLARE_PERSISTENCE_HEURISTIC_DAYS

is_gas_flare_pattern = is_industrial_row & near_gas_oil & high_persistence

df_labeled["industrial_subtype"] = np.where(
    is_gas_flare_pattern,
    "gas_flare_pattern",
    np.where(is_industrial_row, "other_industrial", None),
)

print("=" * 70)
print("INDUSTRIAL SUBTYPE DIAGNOSTIC (reporting only, not a trained label)")
print("=" * 70)
print(f"Total industrial-labeled rows: {int(is_industrial_row.sum()):,}")
print(
    f"...matching gas-flare-like pattern (near WRI gas/oil AND "
    f"active_days_90d >= {GAS_FLARE_PERSISTENCE_HEURISTIC_DAYS}): "
    f"{int(is_gas_flare_pattern.sum()):,} "
    f"({is_gas_flare_pattern.sum() / max(1, is_industrial_row.sum()) * 100:.2f}% "
    "of industrial rows)"
)
print(
    f"Unique H3 cells: "
    f"{df_daily.loc[is_gas_flare_pattern, 'h3_08'].nunique():,}"
)
print(
    "\nThis column is descriptive only — it does not feed the model and "
    "does not change 'label'. It exists to show a flare-like industrial "
    "sub-population is identifiable, in case a judge asks how gas flares "
    "specifically are handled given they are not a standalone class."
)


INDUSTRIAL SUBTYPE DIAGNOSTIC (reporting only, not a trained label)
Total industrial-labeled rows: 450,553
...matching gas-flare-like pattern (near WRI gas/oil AND active_days_90d >= 75): 2,073 (0.46% of industrial rows)
Unique H3 cells: 7

This column is descriptive only — it does not feed the model and does not change 'label'. It exists to show a flare-like industrial sub-population is identifiable, in case a judge asks how gas flares specifically are handled given they are not a standalone class.
